# ReelSense — Model Development

Build the audience-rating regression pipeline with preprocessing fitted only on training data. Do not include `audience_rating` in the feature matrix.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

df = pd.read_csv(Path('../data/Rotten_Tomatoes_Movies3.csv'))
df = df.dropna(subset=['audience_rating']).copy()

numeric = ['runtime_in_minutes','tomatometer_rating','tomatometer_count']
categorical = ['genre','directors','writers','studio_name']
X = df[numeric + categorical].copy()
y = df['audience_rating'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## Baseline structured-data pipeline

In [ ]:
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric), ('cat', categorical_pipe, categorical)])
models = {
    'ridge': Ridge(alpha=10.0),
    'random_forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    print(name, 'trained')

## Text experiment

A separate experiment should integrate TF-IDF from real rows of `critics_consensus` and/or `movie_info`. Avoid the original one-sample TF-IDF demonstration: the vectorizer must be fitted on training text only and evaluated through the same validation discipline as the other features.

## Benchmark table

Record actual MAE, RMSE and R² after execution. Never enter placeholder values or results from a different dataset.